# Wordle: Gemma + LoRA + ES

Train a **linear head** on a Hugging Face causal LM with **Evolution Strategies**. This Week 11 notebook keeps the full **`google/gemma-3-1b-it`** path available, but now defaults to a **smoke-test profile** so you can verify the workflow quickly before committing to a long run. Switch `RUN_PROFILE` in §2 to move from fast validation to the fuller Gemma setup.

**Also:** Richer prompts with **structured constraints** from feedback; optional **supervised warm-start** (cross-entropy to the secret word; label only, not in the prompt).

**Requires:** `torch`, `transformers>=4.50.0`, `jinja2>=3.1.0` for chat-template models such as Gemma, `numpy`, `matplotlib`. **`pip install peft`** only if `USE_LORA=True` in §2. First run downloads HF weights.

## 1. Environment and imports

`peft` is **not** imported here (only needed for `USE_LORA=True`). Install it yourself if you enable LoRA.

**If you see `apply_chat_template requires jinja2>=3.1.0`:** run `python -m pip install -U 'jinja2>=3.1.0'` in the active notebook environment, then restart the kernel.

**If you see `unexpected keyword argument 'normalize_gradient'`:** restart the kernel and run this cell again, or rely on `importlib.reload` below (re-run this cell after editing `src/es_wordle.py`).

In [1]:
import os
import sys
import importlib
from pathlib import Path

import numpy as np
import torch

# Fall back to standard HF downloads; Xet caused import/download issues in this env.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

# Repo root + src on path (works from `notebooks/` or project root)
_here = Path.cwd().resolve()
ROOT = _here.parent if _here.name == "notebooks" else _here
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wordle_env import load_wordle_environment
from wordle_gpt2_policy import WordleGPT2Policy, TRANSFORMERS_AVAILABLE

# Always pick up the latest es_wordle.py (Jupyter caches imports)
import es_wordle
importlib.reload(es_wordle)
from es_wordle import train_es_wordle, train_curriculum

import inspect
_sig = inspect.signature(train_es_wordle)
for _name in (
    "normalize_gradient",
    "eval_deterministic",
    "antithetic",
    "common_random_numbers",
    "ema_beta",
):
    if _name not in _sig.parameters:
        raise RuntimeError(
            f"src/es_wordle.py is missing {_name}. Re-run the import cell after `importlib.reload`, or pull the latest repo."
        )

if not TRANSFORMERS_AVAILABLE:
    raise ImportError("Install transformers: pip install transformers")


def choose_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps_backend = getattr(torch.backends, "mps", None)
    if mps_backend is not None and mps_backend.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def default_model_load_kwargs(device: torch.device) -> tuple[dict, str]:
    if device.type != "cuda":
        return {}, "float32"
    if torch.cuda.is_bf16_supported():
        return {"torch_dtype": torch.bfloat16}, "bfloat16"
    return {"torch_dtype": torch.float16}, "float16"


def _parse_version_tuple(version: str) -> tuple[int, ...]:
    parts = []
    for chunk in version.split("."):
        digits = ""
        for ch in chunk:
            if ch.isdigit():
                digits += ch
            else:
                break
        if not digits:
            break
        parts.append(int(digits))
    return tuple(parts)


def require_chat_template_support() -> None:
    try:
        import jinja2
    except ImportError as exc:
        raise ImportError(
            "Chat-template models require `jinja2>=3.1.0`. Install or upgrade it in the active notebook environment with `python -m pip install -U 'jinja2>=3.1.0'`, then restart the kernel."
        ) from exc

    installed = getattr(jinja2, "__version__", "0")
    if _parse_version_tuple(installed) < (3, 1, 0):
        raise ImportError(
            f"Chat-template models require `jinja2>=3.1.0`, but found {installed}. Upgrade it in the active notebook environment with `python -m pip install -U 'jinja2>=3.1.0'`, then restart the kernel."
        )


print("ROOT:", ROOT)
print("es_wordle:", es_wordle.__file__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("HF_HUB_DISABLE_XET:", os.environ["HF_HUB_DISABLE_XET"])
if torch.cuda.is_available():
    print("cuda_device:", torch.cuda.get_device_name(0))

ROOT: /home/ubuntu/STAT-4830-project-base
es_wordle: /home/ubuntu/STAT-4830-project-base/src/es_wordle.py
torch: 2.11.0+cu130 | cuda: True
HF_HUB_DISABLE_XET: 1
cuda_device: NVIDIA A100 80GB PCIe


## 2. Hyperparameters

The next cell now supports two profiles:

- **`RUN_PROFILE="smoke"`**: fast validation path using a smaller model and short ES/warm-start settings.
- **`RUN_PROFILE="gemma_full"`**: restores the fuller Week 11 Gemma configuration for a real run.

Additional notes:

- **`MODEL_LOAD_KWARGS`**: automatically picks a GPU-friendly dtype on CUDA (`bfloat16` when supported, else `float16`). CPU stays on `float32`.
- **`USE_CHAT_TEMPLATE`**: stays enabled for Gemma instruction checkpoints and disabled for the smoke-test GPT-2 path.
- **`MOCK_ENV`**: remains `True` by default so validation runs stay bounded and always have the target word in the action set.
- **`N_POP` / `N_ITERATIONS` / `WARM_START_STEPS`**: are profile-controlled so the smoke path stays quick while the Gemma path keeps the heavier settings. `N_ITERATIONS` is now interpreted as iterations **per curriculum stage**.
- **`vocab_schedule`**: per-profile list of action-set sizes. `train_curriculum` runs ES at each size in turn, growing both the policy head and the env's secret pool together (see §5). The env is built with `target_pool=policy.words` so every episode is in-principle winnable. Use a single-element schedule (e.g. `[N]`) to disable curriculum.
- **`USE_LORA`**: still defaults to `False` so validation only trains the head unless you explicitly enable adapters.

In [2]:
# --- Hyperparameters (switch RUN_PROFILE for longer runs) ---

SEED = 42
RUN_PROFILE = "gemma_full"  # "smoke" or "gemma_full"
DEVICE = choose_device()

MOCK_ENV = False
USE_PRIME_TARGETS = True
USE_LORA = True  # Leave False for validation; enable only if you want PEFT adapters.
# Single rank only (was a [1, 2, 4] sweep). Prior random-walk drift at r in {64, 128}
# was a population-size problem, not a capacity problem; sweeping tiny ranks is 3x
# wall-clock for ~no information, and the saved compute funds the post-warm-start
# diagnostic + the late-stage iter reshape.
LORA_R = 2
RICHER_PROMPT = True
WARM_START_LR = 3e-4
SIGMA = 0.02
# With NORMALIZE_GRADIENT=False the raw ES ‖ĝ‖ is ~1e4 at this param count, so
# ALPHA is now an actual learning rate (not a fixed step length). α = 3e-6 gives
# an initial step norm ≈ 0.13 — comparable to the old fixed step — and the EMA
# momentum lets the effective step grow when successive gradients agree.
# This likely needs empirical retuning per run; watch `train_grad_cos` and Step‖.
ALPHA = 1.26e-05
NORMALIZE_GRADIENT = False  # Let weak-signal steps be small and strong-signal steps large.
RANK_FITNESS = True
# ES SNR upgrades: antithetic pairs + common random numbers reduce per-iteration
# gradient variance for free (same wall-clock); EMA momentum lets the persistent
# signal accumulate across iterations instead of diffusing.
ANTITHETIC = True            # Requires N_POP to be even (gemma_full N_POP=16 is fine).
COMMON_RANDOM_NUMBERS = True  # Same secret words + sampling for every member in an iter.
EMA_BETA = 0.9               # 0.0 disables momentum; 0.9 is a safe Adam-like default.
EVAL_DETERMINISTIC = True  # greedy eval: Success % reflects argmax policy (clearer after warm-start)
FITNESS_OBJECTIVE = "win_plus_return"
WIN_FITNESS_SCALE = 8.0

PROFILE_CONFIGS = {
    "smoke": {
        "model_name": "distilgpt2",
        "use_chat_template": False,
        "chat_generation_prompt": False,
        "max_prompt_length": 256,
        "vocab_schedule": [8],
        "n_eval_episodes": 1,
        "eval_n_episodes": 4,
        "eval_every": 1,
        "warm_start_steps": 12,
        "n_pop": 4,
        "n_iterations": 2,
        "num_train_examples": 128,
        "num_eval_examples": 16,
    },
    "gemma_full": {
        "model_name": "google/gemma-3-1b-it",
        "use_chat_template": True,
        "chat_generation_prompt": True,
        "max_prompt_length": 512,
        "vocab_schedule": [16, 32, 64, 96, 128, 256, 512, 1024],
        # CRN + n_eval=2 had effectively ~2 secrets discriminating all 16 members;
        # bumping to 4 roughly doubles the rank ordering's effective sample size.
        "n_eval_episodes": 4,
        "eval_n_episodes": 50,
        "eval_every": 1,
        "warm_start_steps": 400,
        "n_pop": 16,
        # Per-stage iters (sums to 160 = old 8x20, same total budget).
        # Biases iters toward the late high-vocab stages where the real
        # generalization problem lives; vocab=16 was solvable in <5 iters anyway.
        "n_iterations": [5, 5, 10, 10, 15, 25, 35, 55],
        "num_train_examples": 2000,
        "num_eval_examples": 20,
    },
}

if RUN_PROFILE not in PROFILE_CONFIGS:
    raise ValueError(f"RUN_PROFILE must be one of {sorted(PROFILE_CONFIGS)}, got {RUN_PROFILE!r}")

cfg = PROFILE_CONFIGS[RUN_PROFILE]
MODEL_NAME = cfg["model_name"]
USE_CHAT_TEMPLATE = cfg["use_chat_template"]
if USE_CHAT_TEMPLATE:
    require_chat_template_support()
CHAT_GENERATION_PROMPT = cfg["chat_generation_prompt"]
MAX_PROMPT_LENGTH = cfg["max_prompt_length"]
N_EVAL_EPISODES = cfg["n_eval_episodes"]
EVAL_N_EPISODES = cfg["eval_n_episodes"]
EVAL_EVERY = cfg["eval_every"]
WARM_START_STEPS = cfg["warm_start_steps"]
N_POP = cfg["n_pop"]
N_ITERATIONS = cfg["n_iterations"]
NUM_TRAIN_EXAMPLES = cfg["num_train_examples"]
NUM_EVAL_EXAMPLES = cfg["num_eval_examples"]

# Mock: action space == secret pool only → every episode solvable; ES / eval see real wins.
if MOCK_ENV:
    from wordle_env import MOCK_WORDLE_TARGETS as _MOCK_T
    _MOCK_ACTIONS = len(_MOCK_T)
else:
    _MOCK_ACTIONS = None

# Curriculum vocabulary schedule: list of action-set sizes, monotonically growing.
# Each stage trains ES at its size, with the env's secret pool restricted to
# `policy.words` so every episode is in-principle winnable. With MOCK_ENV=True
# the schedule collapses to the mock action set so behaviour matches earlier
# notebooks.
if MOCK_ENV:
    VOCAB_SCHEDULE = [_MOCK_ACTIONS]
else:
    VOCAB_SCHEDULE = list(cfg["vocab_schedule"])

if not VOCAB_SCHEDULE:
    raise ValueError("vocab_schedule must contain at least one stage size.")
if any(b < a for a, b in zip(VOCAB_SCHEDULE, VOCAB_SCHEDULE[1:])):
    raise ValueError(f"vocab_schedule must be non-decreasing, got {VOCAB_SCHEDULE!r}")

INITIAL_VOCAB = VOCAB_SCHEDULE[0]
MAX_VOCAB = VOCAB_SCHEDULE[-1]
MODEL_LOAD_KWARGS, MODEL_DTYPE_NAME = default_model_load_kwargs(DEVICE)

# Per-stage warm-start budget. Scaling linearly with vocab fixes the
# "50 Adam steps cannot fit a 1024-way head" failure mode visible in
# previous runs (warm-start loss actually rose across stages 6-8).
# Floor at WARM_START_STEPS so the smallest stage still gets the
# original budget; cap at WARM_START_STEPS * MAX_WS_SCALE to bound
# wall-clock on the largest stages.
MAX_WS_SCALE = 16  # 25600 episodes max per stage at WARM_START_STEPS=400
WARM_START_STEPS_PER_STAGE = [
    min(
        WARM_START_STEPS * MAX_WS_SCALE,
        max(WARM_START_STEPS, int(round(WARM_START_STEPS * v / VOCAB_SCHEDULE[0]))),
    )
    for v in VOCAB_SCHEDULE
]
# Use feedback-consistent words for the random pre-play in warm-start so the
# supervised target is "given these constraints, the secret is X" instead of
# "given garbage prefix, the secret is X". Highest-leverage change for late stages.
WARM_START_FEEDBACK_CONSISTENT = True

import random

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = DEVICE
if isinstance(N_ITERATIONS, int):
    _iters_desc = f"N_ITERATIONS={N_ITERATIONS} per stage (total={N_ITERATIONS * len(VOCAB_SCHEDULE)})"
else:
    _iters_desc = f"iters_per_stage={list(N_ITERATIONS)} (total={sum(N_ITERATIONS)})"
print(
    f"Profile: {RUN_PROFILE} | Model: {MODEL_NAME} | chat_template: {USE_CHAT_TEMPLATE} | LoRA: {USE_LORA} (r={LORA_R})\n"
    f"  device={device}  model_dtype={MODEL_DTYPE_NAME}  MAX_PROMPT_LENGTH={MAX_PROMPT_LENGTH}\n"
    f"  vocab_schedule={VOCAB_SCHEDULE}  (initial={INITIAL_VOCAB}, max={MAX_VOCAB})\n"
    f"  ES: N_POP={N_POP}  {_iters_desc}  n_eval_episodes={N_EVAL_EPISODES}  "
    f"eval_n_episodes={EVAL_N_EPISODES}\n"
    f"  warm-start eps per stage: {WARM_START_STEPS_PER_STAGE} "
    f"(total={sum(WARM_START_STEPS_PER_STAGE)}, feedback_consistent={WARM_START_FEEDBACK_CONSISTENT})"
)
if MODEL_LOAD_KWARGS:
    print("model_load_kwargs:", MODEL_LOAD_KWARGS)

# Heads-up on ALPHA calibration: the rationale comment above (initial step ~ 0.13)
# was measured at LORA_R=64 with ~764k trainable params. With the rank-fitness
# ES gradient norm scaling like sqrt(n_params/N_POP)/sigma, dropping to LORA_R=2
# shrinks the trainable count ~10x and the initial step ~3x. If `Step‖` in the
# first iteration prints noticeably smaller than ~0.13 and `‖θ-θ₀‖` plateaus
# below ~0.5 in stage 1, ALPHA needs to be bumped (try ~1e-5 first).
print(
    f"  reminder: ALPHA={ALPHA:.1e} was tuned at LORA_R=64; with LORA_R={LORA_R} the\n"
    f"  initial step will be smaller. Watch the iter-0 Step‖ in stage 1 and bump ALPHA\n"
    f"  if it's much below ~0.13."
)

Profile: gemma_full | Model: google/gemma-3-1b-it | chat_template: True | LoRA: True (r=2)
  device=cuda  model_dtype=bfloat16  MAX_PROMPT_LENGTH=512
  vocab_schedule=[16, 32, 64, 96, 128, 256, 512, 1024]  (initial=16, max=1024)
  ES: N_POP=16  iters_per_stage=[5, 5, 10, 10, 15, 25, 35, 55] (total=160)  n_eval_episodes=4  eval_n_episodes=50
  warm-start eps per stage: [400, 800, 1600, 2400, 3200, 6400, 6400, 6400] (total=27600, feedback_consistent=True)
model_load_kwargs: {'torch_dtype': torch.bfloat16}
  reminder: ALPHA=1.3e-05 was tuned at LORA_R=64; with LORA_R=2 the
  initial step will be smaller. Watch the iter-0 Step‖ in stage 1 and bump ALPHA
  if it's much below ~0.13.


## 3. Build policy and environment (no-op)

This section used to allocate a `WordleGPT2Policy` + env that cell 5 (`Run ES training`) immediately threw away and rebuilt — that meant downloading Gemma weights twice and ~1 GB of redundant GPU allocation. The assertion that mock-targets are present in the action set and the trainable-parameter printout now live in cell 5 instead.

Skip the cell below; it just prints a note. The actual policy + env are constructed once in cell 5.

In [3]:
# Intentionally empty: cell 10 builds the policy + env that the curriculum run
# actually uses. Building them here too would download Gemma weights twice and
# allocate ~1 GB of GPU memory that gets immediately freed. The mock-target
# coverage assertion and the param-count printout that used to live here are
# folded into cell 10 instead.
print("Cell 3 is a no-op; the policy + env are built in cell 5 (`Run ES training`).")

Cell 3 is a no-op; the policy + env are built in cell 5 (`Run ES training`).


## 4. Optional supervised warm-start

Random play for 1–4 guesses, then **cross-entropy** toward the secret word (the answer is **not** in the prompt). Set `WARM_START_STEPS = 0` to skip.

In [4]:
# from wordle_gpt2_warmstart import supervised_warm_start_wordle

# if WARM_START_STEPS > 0:
#     ws = supervised_warm_start_wordle(
#         policy,
#         env,
#         n_steps=WARM_START_STEPS,
#         lr=WARM_START_LR,
#         device=device,
#         seed=SEED,
#         verbose=True,
#     )
#     print("Warm-start: fitted", len(ws["loss"]), "steps; skipped", ws["skipped"])
# else:
#     print("Skipping warm-start.")


## 5. Run ES training

`train_es_wordle` prints **every iteration** when `verbose=True`.

- **`‖θ−θ₀‖`**: L2 distance of trainable weights from init — should grow every step (proof the optimizer is moving parameters).
- **`Step‖`**: actual update norm. With `NORMALIZE_GRADIENT=False` (default here) this now *varies* with signal strength — small when ES is noisy, large when EMA momentum finds agreement across iterations. If `NORMALIZE_GRADIENT=True` it's a fixed `α` by design.
- **`cos(ĝ)`**: cosine similarity between the current and previous *raw* ES gradient. This is the cleanest "is there real signal?" diagnostic — pure diffusion gives `~0`, coherent descent gives clearly positive values (typically `> 0.05`).
- **`popσ`**: spread of fitness across the ES population (noise level).

`history` now includes **`train_iter`, `train_fitness`, `param_drift`, `pop_fitness_std`, `train_grad_cos`, …** every iteration (not only eval checkpoints). The plot cell uses them in the **bottom row**.

Full eval lines (Success / Eval Reward) appear when `iteration % EVAL_EVERY == 0` or on the last iteration.

**New ES SNR upgrades active in this run (cheap per iteration):**

- `ANTITHETIC=True` — uses `N_POP / 2` mirrored pairs `(+ε, −ε)`, roughly halving gradient variance at identical wall-clock.
- `COMMON_RANDOM_NUMBERS=True` — every population member in an iteration sees the same secret words + sampling draws, so rank comparisons isolate the *policy* difference rather than env/sampling noise.
- `EMA_BETA=0.9` — Adam-style momentum on the ES gradient with bias correction. Lets consistent directions accumulate across iterations; diffusion averages to zero.

In [5]:
import gc
import random

from wordle_gpt2_warmstart import supervised_warm_start_wordle

# Single-rank run (was a [1, 2, 4] sweep). Prior run's random-walk drift at
# r in {64, 128} was a population-size / SNR problem, not a capacity problem;
# sweeping tiny ranks at fixed N=16 was 3x wall-clock for ~no information.
# The saved compute funds n_eval_episodes 2->4, the late-stage iter reshape,
# and the post-warm-start diagnostic.

if not USE_LORA:
    raise ValueError("LoRA single-rank run expects USE_LORA=True in hyperparameters.")

# Cell 3/4 may have created and warm-started a single-run policy.
# Drop it before this cell creates its own fresh policy + env.
if "policy" in globals():
    del policy
if "ws" in globals():
    del ws
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

run_seed = int(SEED)
random.seed(run_seed)
np.random.seed(run_seed)
torch.manual_seed(run_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(run_seed)

print(f"\n=== LoRA rank {LORA_R} (seed={run_seed}) ===")

policy = WordleGPT2Policy(
    model_name=MODEL_NAME,
    use_prime_targets=USE_PRIME_TARGETS,
    max_vocab_size=INITIAL_VOCAB,
    max_prompt_length=MAX_PROMPT_LENGTH,
    include_mock_targets_in_vocab=True,
    richer_prompt=RICHER_PROMPT,
    use_chat_template=USE_CHAT_TEMPLATE,
    chat_generation_prompt=CHAT_GENERATION_PROMPT,
    use_lora=USE_LORA,
    lora_r=LORA_R,
    model_kwargs=MODEL_LOAD_KWARGS,
).to(device)

# Folded in from the (now-empty) cell 3: sanity-check coverage of the mock
# targets in the action set and print the trainable-param count, since the
# raw ES gradient norm scales with sqrt(n_params/N_POP) and that's what
# determines whether the current ALPHA produces a meaningful step.
from wordle_env import MOCK_WORDLE_TARGETS
assert all(w in policy.words for w in MOCK_WORDLE_TARGETS), (
    "MOCK_WORDLE_TARGETS missing from policy.words; ES would generate target "
    "words the policy cannot emit."
)
print(f"Trainable (ES): {policy.count_trainable_parameters():,}")
print(f"Total params:   {policy.count_parameters():,}")
print(f"Action dim:     {policy.action_dim} (initial; will grow to {MAX_VOCAB} across curriculum)")
print(f"Chat template:  {policy.use_chat_template}")
print(f"Load kwargs:    {MODEL_LOAD_KWARGS or {'torch_dtype': 'float32'}}")

# Env target pool starts aligned with the initial policy vocab; curriculum
# stages call env.set_target_pool(policy.words) as the head grows.
env = load_wordle_environment(
    num_train_examples=NUM_TRAIN_EXAMPLES,
    num_eval_examples=NUM_EVAL_EXAMPLES,
    use_prime_intellect=not MOCK_ENV,
    target_pool=policy.words,
)

# --- One-time ALPHA calibration at init ---
# The cell-2 reminder warns that ALPHA was tuned at LORA_R=64 (~764k trainable);
# at the current LORA_R the rank-fitness raw ‖ĝ‖ scales like
# sqrt(n_params/N_POP)/sigma, so the initial step (~0.13 at the calibration
# point) shrinks. We probe the initial gradient norm with a single ES estimate
# and report what ALPHA *would* be needed to recover step ~ 0.13. We do NOT
# overwrite ALPHA — surface the suggestion only.
#
# RNG state is snapshotted/restored so the calibration probe does not perturb
# the training run's stream of secrets / sampling draws.
from es_wordle import es_gradient_estimate_wordle

_target_step = 0.13
_min_step = 0.05

_py_state = random.getstate()
_np_state = np.random.get_state()
_torch_state = torch.get_rng_state()
_cuda_state = (
    torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
)
try:
    _grad, _avg_fit, _fits, _avg_win, _pop_std = es_gradient_estimate_wordle(
        policy,
        env,
        N=N_POP,
        sigma=SIGMA,
        n_eval_episodes=N_EVAL_EPISODES,
        max_turns=6,
        rank_fitness=RANK_FITNESS,
        fitness_objective=FITNESS_OBJECTIVE,
        win_fitness_scale=WIN_FITNESS_SCALE,
        antithetic=ANTITHETIC,
        common_random_numbers=COMMON_RANDOM_NUMBERS,
    )
    _g_norm = float(_grad.norm().item())
    _implied_step = ALPHA * _g_norm
    print(
        f"\n[ALPHA calibration @ init, vocab={INITIAL_VOCAB}, "
        f"trainable={policy.count_trainable_parameters():,}]\n"
        f"  raw ‖ĝ‖           = {_g_norm:.4g}\n"
        f"  ALPHA * ‖ĝ‖       = {_implied_step:.4g}   (target ≈ {_target_step})\n"
        f"  ES probe avg_fit  = {_avg_fit:+.3f}   ES_win = {_avg_win:.1%}   popσ = {_pop_std:.4f}"
    )
    if _implied_step < _min_step and _g_norm > 1e-8:
        _suggested_alpha = _target_step / _g_norm
        print(
            f"  ⚠ implied step {_implied_step:.4g} < {_min_step} threshold.\n"
            f"    Suggested ALPHA ≈ {_suggested_alpha:.2e} would give step ≈ {_target_step}.\n"
            f"    (Not overwriting; edit cell 4 if you agree.)"
        )
    else:
        print(f"  implied step within healthy range; keeping ALPHA={ALPHA:.2e}.")
finally:
    # Restore RNG so the calibration probe doesn't shift the training stream.
    random.setstate(_py_state)
    np.random.set_state(_np_state)
    torch.set_rng_state(_torch_state)
    if _cuda_state is not None:
        torch.cuda.set_rng_state_all(_cuda_state)
    del _py_state, _np_state, _torch_state, _cuda_state
    del _grad, _avg_fit, _fits, _avg_win, _pop_std
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

history = train_curriculum(
    policy,
    env,
    vocab_schedule=VOCAB_SCHEDULE,
    n_iterations_per_stage=N_ITERATIONS,
    warm_start_fn=supervised_warm_start_wordle,
    # Per-stage budget (list, not int): late stages with N=512/1024 actions need
    # far more Adam steps than the previous flat budget of 50 opt steps could
    # provide. See WARM_START_STEPS_PER_STAGE in cell 2 for the schedule.
    warm_start_steps=WARM_START_STEPS_PER_STAGE,
    warm_start_kwargs={
        "lr": WARM_START_LR,
        "device": device,
        "seed": run_seed,  # train_curriculum adds 1000 * stage_idx per stage
        "verbose": True,
        # Batched warm-start: 8 examples per opt step, ~6-8x wall-clock cut on
        # the warm-start budget for Gemma-3-1b. Requires policy.forward_logits_batch
        # (WordleGPT2Policy provides it).
        "batch_size": 8,
        # Draw random pre-play from words consistent with accumulated feedback
        # so the supervised target is "given these constraints, predict X"
        # rather than "given garbage prefix, predict random secret". Largest
        # impact on late-stage warm-start fit.
        "feedback_consistent_random": WARM_START_FEEDBACK_CONSISTENT,
    },
    # Greedy post-warm-start eval per stage. Compare against eval_success at
    # the same stage's end (history["eval_success"]) to attribute progress
    # between warm-start and ES — the single sharpest "is ES doing anything?"
    # diagnostic for this whole pipeline.
    post_warm_start_eval_episodes=EVAL_N_EPISODES,
    post_warm_start_eval_deterministic=EVAL_DETERMINISTIC,
    N=N_POP,
    sigma=SIGMA,
    alpha=ALPHA,
    n_eval_episodes=N_EVAL_EPISODES,
    max_turns=6,
    eval_every=EVAL_EVERY,
    verbose=True,
    normalize_gradient=NORMALIZE_GRADIENT,
    eval_n_episodes=EVAL_N_EPISODES,
    rank_fitness=RANK_FITNESS,
    eval_deterministic=EVAL_DETERMINISTIC,
    fitness_objective=FITNESS_OBJECTIVE,
    win_fitness_scale=WIN_FITNESS_SCALE,
    antithetic=ANTITHETIC,
    common_random_numbers=COMMON_RANDOM_NUMBERS,
    ema_beta=EMA_BETA,
)

final_success = float(history["eval_success"][-1])
print(f"\nFinal success probability (rank={LORA_R}): {final_success:.3f}")

# Per-stage attribution: warm-start contribution vs ES contribution.
# If post_warmstart_success == eval_success at the same stage end, ES did nothing
# for that stage. If consistently true across stages, warm-start is carrying the run.
print("\nPer-stage warm-start vs ES attribution:")
print(f"  {'stage':>5} {'vocab':>6} {'post_WS':>8} {'end_ES':>8} {'delta':>8}")
_eval_iters = list(history.get("iteration", []))
_eval_succs = list(history.get("eval_success", []))
_stage_starts = list(history.get("stage_starts", []))
_stage_vocab = list(history.get("stage_vocab_sizes", []))
_post_ws = list(history.get("post_warmstart_success", []))
for s_idx, (s_start, s_vocab, post_ws) in enumerate(zip(_stage_starts, _stage_vocab, _post_ws)):
    s_end = _stage_starts[s_idx + 1] - 1 if s_idx + 1 < len(_stage_starts) else (
        _eval_iters[-1] if _eval_iters else s_start
    )
    end_es = float("nan")
    for it, succ in zip(_eval_iters, _eval_succs):
        if s_start <= it <= s_end:
            end_es = float(succ)
    delta = end_es - post_ws if (end_es == end_es and post_ws == post_ws) else float("nan")
    print(f"  {s_idx + 1:>5} {s_vocab:>6} {post_ws:>8.1%} {end_es:>8.1%} {delta:>+8.1%}")


=== LoRA rank 2 (seed=42) ===
[INFO] Using bundled Wordle answer list (2315 words); Prime Intellect not available (ModuleNotFoundError).


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Trainable (ES): 391,184
Total params:   1,000,277,136
Action dim:     16 (initial; will grow to 1024 across curriculum)
Chat template:  True
Load kwargs:    {'torch_dtype': torch.bfloat16}
[OK] Local Wordle environment ready (16 target words from explicit target_pool).

[ALPHA calibration @ init, vocab=16, trainable=391,184]
  raw ‖ĝ‖           = 1.035e+04
  ALPHA * ‖ĝ‖       = 0.1304   (target ≈ 0.13)
  ES probe avg_fit  = +4.333   ES_win = 35.9%   popσ = 1.8104
  implied step within healthy range; keeping ALPHA=1.26e-05.

=== Curriculum stage 1/8 | vocab: 16 -> 16 (+0) | iters: 5 | warm-start eps: 400 ===
  warm-start 80/400 | loss=3.0269 | opt_steps=10 | bs=8 | skipped=0
  warm-start 160/400 | loss=2.9568 | opt_steps=20 | bs=8 | skipped=0
  warm-start 240/400 | loss=2.9068 | opt_steps=30 | bs=8 | skipped=0
  warm-start 320/400 | loss=2.8943 | opt_steps=40 | bs=8 | skipped=0
  warm-start 400/400 | loss=2.8828 | opt_steps=50 | bs=8 | skipped=0
Warm-start (stage 1, vocab=16): fitted 50

KeyboardInterrupt: 

## 6. Plot training curves

In [6]:
import matplotlib.pyplot as plt

it = history["iteration"]
ti = history["train_iter"]
fig, axes = plt.subplots(2, 4, figsize=(16, 6))

axes[0, 0].plot(it, history["eval_reward"], "b-o", ms=3)
axes[0, 0].set_title("Eval mean reward")
axes[0, 0].set_xlabel("iteration")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(it, history["eval_success"], "g-o", ms=3)
axes[0, 1].set_title("Eval success rate")
axes[0, 1].set_xlabel("iteration")
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(it, history["eval_turns"], "m-o", ms=3)
axes[0, 2].set_title("Mean turns (eval)")
axes[0, 2].set_xlabel("iteration")
axes[0, 2].grid(True, alpha=0.3)

axes[0, 3].plot(ti, history["train_grad_norm"], "r-", lw=1)
axes[0, 3].set_title("Applied ‖ĝ‖ (post-EMA)")
axes[0, 3].set_xlabel("iteration")
axes[0, 3].grid(True, alpha=0.3)

axes[1, 0].plot(ti, history["param_drift"], "k-", lw=1)
axes[1, 0].set_title("‖θ − θ₀‖ (params are updating)")
axes[1, 0].set_xlabel("iteration")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(ti, history["train_fitness"], "c-", lw=1, alpha=0.8)
axes[1, 1].set_title("ES mean fitness (every iter)")
axes[1, 1].set_xlabel("iteration")
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(ti, history["pop_fitness_std"], "orange", lw=1)
axes[1, 2].set_title("Population fitness σ (spread)")
axes[1, 2].set_xlabel("iteration")
axes[1, 2].grid(True, alpha=0.3)

# cos(ĝ_t, ĝ_{t-1}) on raw (pre-EMA) ES gradients. Healthy CRN run is
# clearly positive (signal beats noise) but well below 1 (secrets vary
# across iterations). Pinned near +1.0 at the start of a stage = a sign
# the CRN advance isn't decorrelating secrets across iterations.
gc_arr = np.array(history["train_grad_cos"], dtype=float)
axes[1, 3].plot(ti, gc_arr, "purple", lw=1)
axes[1, 3].axhline(0.0, color="gray", lw=0.6, ls=":")
axes[1, 3].set_title("cos(ĝ_t, ĝ_{t−1})  (raw)")
axes[1, 3].set_xlabel("iteration")
axes[1, 3].set_ylim(-1.05, 1.05)
axes[1, 3].grid(True, alpha=0.3)

# Overlay curriculum stage boundaries on every subplot (skip the first stage's
# marker at iteration 0 to avoid clutter at the y-axis).
stage_starts = history.get("stage_starts", [])
stage_vocab_sizes = history.get("stage_vocab_sizes", [])
for ax in axes.flat:
    for s_idx, s_iter in enumerate(stage_starts):
        if s_iter <= 0:
            continue
        ax.axvline(s_iter, color="gray", lw=0.8, ls="--", alpha=0.5)
        if ax is axes[0, 1] and s_idx < len(stage_vocab_sizes):
            ax.text(
                s_iter, 1.02, f"N={stage_vocab_sizes[s_idx]}",
                fontsize=7, color="gray", ha="left", va="bottom",
            )

rank_note = f" | LoRA r={LORA_R}" if USE_LORA else ""
schedule_note = (
    f" | curriculum {stage_vocab_sizes}" if len(stage_vocab_sizes) > 1 else ""
)
plt.suptitle(f"Wordle ES | {MODEL_NAME} + head{rank_note}{schedule_note}")
plt.tight_layout()
plt.show()

mkdir -p failed for path /home/ubuntu/.config/matplotlib: [Errno 13] Permission denied: '/home/ubuntu/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-yqif5rq9 because there was an issue with the default path (/home/ubuntu/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


NameError: name 'history' is not defined

## 7. Save checkpoint (optional)

Saves the **head** plus `words` and `history` under `models/` at the repo root when `USE_LORA=False`.

When `USE_LORA=True`, checkpoint saving is skipped to avoid writing large LoRA model files.

In [ ]:
if USE_LORA:
    print(f"LoRA rank used for this run: r={LORA_R}")
    print("Skipping checkpoint save for LoRA run to avoid large model artifacts.")
else:
    save_path = ROOT / "models" / f"wordle_gemma_es_head.{RUN_PROFILE}.pt"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "head": policy.head.state_dict(),
        "model_name": MODEL_NAME,
        "words": policy.words,
        "history": history,
        "use_lora": USE_LORA,
        "run_profile": RUN_PROFILE,
        "model_load_kwargs": {k: str(v) for k, v in MODEL_LOAD_KWARGS.items()},
    }
    if getattr(policy, "_lm_trainable", False):
        payload["lm"] = policy.lm.state_dict()
    torch.save(payload, save_path)
    print("Saved:", save_path)